# Example 1.4 / 2: Indexed Constraints

Companion notebook for **1.4 Constraints** in the AMPL lab Documentation, section "Indexed Constraints". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Indexed Constraints

Most constraints come in families. With an indexing expression, one declaration generates one
constraint per member of a set. Here production passes through two stages, each with its own
hours. From Python, the two-dimensional `rate` is a dictionary keyed by `(product, stage)`
tuples.

To see the constraints AMPL generated, use AMPL's `expand` command: from Python,
`ampl.get_output("expand Time;")` runs it and returns what it prints as a string.
After the solve, `ampl.var["Make"].to_pandas()` gives the values of `Make` as a pandas
DataFrame, and `get_values(["body"])` gives the `body` of every member of `Time`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
set STAGE;
param rate {PROD, STAGE} > 0;    # tons per hour in each stage
param avail {STAGE} >= 0;        # hours available in each stage
param profit {PROD};
param market {PROD} >= 0;
var Make {p in PROD} >= 0, <= market[p];

maximize Total_Profit: sum {p in PROD} profit[p] * Make[p];

subject to Time {s in STAGE}:
    sum {p in PROD} (1 / rate[p,s]) * Make[p] <= avail[s];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils"]
ampl.set["STAGE"] = ["reheat", "roll"]
ampl.param["rate"] = {("bands", "reheat"): 200, ("bands", "roll"): 200,
                      ("coils", "reheat"): 200, ("coils", "roll"): 140}
ampl.param["avail"] = {"reheat": 35, "roll": 40}
ampl.param["profit"] = {"bands": 25, "coils": 30}
ampl.param["market"] = {"bands": 6000, "coils": 4000}

In [5]:
# run.py
print(ampl.get_output("expand Time;"))

ampl.solve(solver="highs")
print(ampl.solve_result)
print(ampl.var["Make"].to_pandas())
print(ampl.con["Time"].get_values(["body"]).to_pandas())

subject to Time['reheat']:
	0.005*Make['bands'] + 0.005*Make['coils'] <= 35;

subject to Time['roll']:
	0.005*Make['bands'] + 0.00714286*Make['coils'] <= 40;


HiGHS 1.14.0: optimal solution; objective 186666.6667
2 simplex iterations
0 barrier iterations
solved
          Make.val
bands  4666.666667
coils  2333.333333
        Time.body
reheat       35.0
roll         40.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils;
set STAGE := reheat roll;
param rate:   reheat  roll :=
  bands         200    200
  coils         200    140 ;
param avail := reheat 35  roll 40;
param:  profit  market :=
  bands    25    6000
  coils    30    4000 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
expand Time;
option solver highs;
solve;
display Make;
display Time.body;

subject to Time['reheat']:
	0.005*Make['bands'] + 0.005*Make['coils'] <= 35;

subject to Time['roll']:
	0.005*Make['bands'] + 0.00714286*Make['coils'] <= 40;

HiGHS 1.14.0: optimal solution; objective 186666.6667
2 simplex iterations
0 barrier iterations
Make [*] :=
bands  4666.67
coils  2333.33
;

Time.body [*] :=
reheat  35
  roll  40
;

